# DS4 Statistical Mechanics (StM) -- Stage 4 Step 1

**Notebook:** `51_ds4_statistical_mechanics.ipynb`  
**Model family:** Statistical Mechanics (StM)  
**Dataset:** DS4 -- Large Naturalistic OneSec Users (N=1,039)  
**Prerequisites:** DS4 StoP (53), DS4 InT (52), DS4 DyS (55) -- all complete, loaded directly, nothing re-estimated.  
**Blueprint:** `outputs/reports/stage4/step1_modeling/ds4_stm_blueprint.md` (see its SS0 re-verification note, written 2026-08-06 before this notebook)

---

## Goal

Implement the rigorous OU-based Boltzmann bridge: beta_OU=2*theta/sigma^2 is exact (not an approximation like DS5's beta_behav=2/sig_hat^2), because the OU stationary distribution really is Gaussian. Resolves DS5 StM's FLAG A.

## Not the goal

- No re-estimation of mu/theta/sigma -- loaded directly from StoP.
- No re-computation of H_StM as an independent quantity -- for DS4 specifically it is the exact same formula as InT's H_SDE, joined directly, not recomputed.
- No phase-transition/T_c sweep beyond a simple simulated one (FLAG 4, deferred detail, same as DS5).

## Core claim

> This notebook resolves DS5 StM's FLAG A. The OU process produces an exact Gaussian stationary distribution and an exact confining quadratic potential. beta_OU=2*theta/sigma^2 is rigorous, not approximate. Every quantity here is built from StoP's **daily**-scale parameters specifically -- StoP's own weekly-aggregation check found a meaningfully different relaxation time at weekly resolution, so beta_OU/chi_OU/H_StM should not be read as timescale-independent.

## Output

```
outputs/model_outputs/ds4_statistical_mechanics/
  participant_ds4_stm_parameters.csv
  ds4_stm_summary.csv
  ds4_ds5_stm_comparison.csv
  model_run_summary.md
  figures/
```


## FLAG decisions

| Flag | Decision |
|---|---|
| FLAG 1 | OU SDE: dX_t=theta(mu-X_t)dt+sigma dW_t; beta_OU=2theta/sigma^2; individual V(x) exact and defensible |
| FLAG 2 (updated) | Reuse StoP's own mu_hat_OU/theta_hat_OU/sigma_hat_OU directly -- do not re-fit |
| FLAG 3 | No condition comparison (DS4 is observational) |
| FLAG 4 | Phase transition T_c estimated from beta_OU distribution (simulated sweep), same as DS5 |
| FLAG 6 (new) | beta_OU/chi_OU/H_StM are daily-timescale specific -- inherits StoP's daily/weekly divergence caveat |


## Proxy warning + DS4 structural caveat

- `X_t` is a behavioral override rate (opened/dismissed within `intervention_event` class), not a latent psychological state -- inherited from StoP.
- No experimental condition in DS4 -- all StM results are within/cross-participant heterogeneity, never a friction "field" h.
- FLAG III (unchanged from original blueprint): individual V(x) is mathematically exact for the OU model, but the OU model itself is an assumption -- label it "OU model-implied energy landscape," not "true behavioral energy landscape."


## Theory 1: OU stationary distribution and exact Boltzmann bridge

$$dX_t = \theta(\mu-X_t)dt + \sigma dW_t \quad\Rightarrow\quad P_\infty(x) = \mathcal{N}(\mu, \sigma^2/2\theta)$$

$$V(x) = \frac{\theta}{2}(x-\mu)^2 \qquad \beta_{OU} = \frac{2\theta}{\sigma^2} \qquad T_{OU} = \frac{\sigma^2}{2\theta} = 1/\beta_{OU}$$

All exact for OU (DS5's beta_behav=2/sig_hat^2 was an approximation -- constant-drift has no true stationary distribution to derive V(x) from).


## Theory 2: two-state Boltzmann model (same apparatus as DS5 StM)

Two states: "use" (opened) and "non-use" (dismissed), energies 0 and $E_{use}$. Given the observed opened-probability $p_{use}$ and the rigorous $\beta_{OU}$, invert the Boltzmann relation for $E_{use}$:

$$p_{use} = \frac{e^{-\beta E_{use}}}{1+e^{-\beta E_{use}}} \;\Rightarrow\; E_{use} = -T_{OU}\ln\!\left(\frac{p_{use}}{1-p_{use}}\right) \qquad Z_{2state} = \frac{1}{1-p_{use}}$$

$$\chi_{OU} = \beta_{OU}\, p_{use}(1-p_{use}) \qquad F_{OU} = -\frac{1}{\beta_{OU}}\ln Z_{2state}$$


## DS5 vs DS4 comparison -- FLAG A resolved

DS5's `beta_behav=2/sig_hat^2` assumes a potential-well SDE that its own constant-drift StoP model does not actually have (FLAG A in `41_ds5_statistical_mechanics.ipynb`). DS4's OU process genuinely has that potential well -- `beta_OU=2*theta/sigma^2` is exact. The two are **not directly comparable in magnitude** (different underlying SDE) but the qualitative question -- does higher noise mean lower behavioral temperature-inverse (beta)? -- can be compared descriptively.


## Theory-to-data mapping

| Quantity | DS4 source |
|---|---|
| mu, theta, sigma | StoP `participant_parameters.csv`, loaded directly |
| beta_OU, T_OU | Derived: 2*theta/sigma^2, sigma^2/2theta |
| p_use | mu_hat_OU (long-run opened-probability) |
| E_use, Z_2state, chi_OU, F_OU | Derived, see Theory 2 |
| V(x) | (theta/2)(x-mu)^2, exact |
| H_StM | InT's `H_SDE`, joined directly (identical formula for DS4) |
| regime_label | DyS's `participant_dynamics_parameters.csv`, joined |


## Equation gaps (paper additions needed)

OU SDE, stationary distribution, V(x)=quadratic, beta_OU=2theta/sigma^2, tau=1/theta, fixed point X*=mu (already added when DyS/InT were implemented) -- none of these are yet in the conceptual paper. See `ds4_stm_blueprint.md` SS4 for the full table.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

STOP_PATH = Path("outputs/model_outputs/ds4_stochastic_processes/participant_parameters.csv")
INT_PATH = Path("outputs/model_outputs/ds4_information_theory/participant_entropy_metrics.csv")
DYS_PATH = Path("outputs/model_outputs/ds4_dynamical_systems/participant_dynamics_parameters.csv")
DS5_STM_PATH = Path("outputs/model_outputs/statistical_mechanics/participant_stm_parameters.csv")
FEAT_PATH = Path("outputs/features/ds4/ds4_behavior_features.csv")
OUT_DIR = Path("outputs/model_outputs/ds4_statistical_mechanics")
FIG_DIR = OUT_DIR / "figures"

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
stop_params = pd.read_csv(STOP_PATH)
int_metrics = pd.read_csv(INT_PATH)
dys_params = pd.read_csv(DYS_PATH)
stage1_feats = pd.read_csv(FEAT_PATH)

stm = stop_params[[
    "participant_id_canonical", "mu_hat_OU", "theta_hat_OU", "sigma_hat_OU",
    "tau_hat", "fit_valid",
]].copy()
stm = stm.merge(int_metrics[["participant_id_canonical", "H_SDE"]].rename(
    columns={"H_SDE": "H_StM"}), on="participant_id_canonical", how="left")
stm = stm.merge(dys_params[["participant_id_canonical", "regime_label"]],
    on="participant_id_canonical", how="left")

print(f"stm base table: {len(stm)} rows, {int(stm['fit_valid'].sum())} valid")


In [ ]:
stm["beta_OU"] = 2 * stm["theta_hat_OU"] / stm["sigma_hat_OU"] ** 2
stm["T_OU"] = stm["sigma_hat_OU"] ** 2 / (2 * stm["theta_hat_OU"])

valid = stm[stm["fit_valid"]]
print(valid[["beta_OU", "T_OU"]].describe())


In [ ]:
stm["p_use"] = stm["mu_hat_OU"].clip(1e-6, 1 - 1e-6)
stm["E_use"] = -stm["T_OU"] * np.log(stm["p_use"] / (1 - stm["p_use"]))
stm["Z_2state"] = 1 / (1 - stm["p_use"])
stm["chi_OU"] = stm["beta_OU"] * stm["p_use"] * (1 - stm["p_use"])
stm["F_OU"] = -(1 / stm["beta_OU"]) * np.log(stm["Z_2state"])

valid = stm[stm["fit_valid"]]
print(valid[["p_use", "E_use", "Z_2state", "chi_OU", "F_OU"]].describe())


In [ ]:
def v_of_x(x, theta, mu):
    return (theta / 2) * (x - mu) ** 2

x_grid = np.linspace(0, 1, 200)
print("V(x) is computed on demand per participant via v_of_x(x, theta_hat_OU, mu_hat_OU) "
      "-- no separate export table needed beyond the parameters already in stm.")


In [ ]:
stm["V_min_location"] = stm["mu_hat_OU"]
stm["V_curvature"] = stm["theta_hat_OU"] / 2
stm["V_depth"] = stm["beta_OU"]

valid = stm[stm["fit_valid"]]
print(valid[["V_min_location", "V_curvature", "V_depth"]].describe())


In [ ]:
valid = stm[stm["fit_valid"]]
bridge_check = valid[["participant_id_canonical", "H_StM"]].merge(
    int_metrics[["participant_id_canonical", "H_emp"]],
    on="participant_id_canonical", how="left",
)
r_hstm_hemp = bridge_check["H_StM"].corr(bridge_check["H_emp"])
print(f"H_StM (=InT's H_SDE) vs H_emp: r={r_hstm_hemp:.4f} (n={len(valid)}) -- "
      f"same bridge InT already reported, reproduced here for the StM record")


In [ ]:
ds5_stm = pd.read_csv(DS5_STM_PATH)

comparison = pd.DataFrame([
    {"dataset": "DS5 (approximate)", "n": int(ds5_stm["beta_behav"].notna().sum()),
     "median_beta": ds5_stm["beta_behav"].median(), "median_T": ds5_stm["T_behav"].median()},
    {"dataset": "DS4 (exact, daily-scale)", "n": int(stm["fit_valid"].sum()),
     "median_beta": valid["beta_OU"].median(), "median_T": valid["T_OU"].median()},
])
print(comparison)
print("\nNote: NOT directly comparable in magnitude (different underlying SDE) -- "
      "reported descriptively only, per blueprint SS6.")


In [ ]:
print("tau_hat (relaxation time, days) among valid fits:")
print(valid["tau_hat"].describe())
print("\nSee StoP's own participant_parameters_weekly_sensitivity.csv for the "
      "weekly-scale comparison -- this notebook does not recompute it.")


In [ ]:
beta_median = valid["beta_OU"].median()
stm["temperature_group"] = pd.NA
stm.loc[stm["fit_valid"] & (stm["beta_OU"] >= beta_median), "temperature_group"] = "low_T_high_beta"
stm.loc[stm["fit_valid"] & (stm["beta_OU"] < beta_median), "temperature_group"] = "high_T_low_beta"
print(f"beta_OU median = {beta_median:.3f}")
print(stm["temperature_group"].value_counts(dropna=False))


In [ ]:
corr_feats = stage1_feats[[
    "participant_id_canonical", "dominant_intervention_type", "activity_persistence",
]].copy()
stm_corr = stm.merge(corr_feats, on="participant_id_canonical", how="left")
valid_corr = stm_corr[stm_corr["fit_valid"]]

r_persistence_beta = valid_corr["activity_persistence"].corr(valid_corr["beta_OU"])
print(f"corr(activity_persistence, beta_OU) = {r_persistence_beta:.4f}")

print("\nbeta_OU by regime_label (from DyS):")
print(valid.groupby(stm.loc[valid.index, "regime_label"], observed=True)["beta_OU"].describe())


In [ ]:
export_cols = [
    "participant_id_canonical", "mu_hat_OU", "theta_hat_OU", "sigma_hat_OU", "tau_hat",
    "beta_OU", "T_OU", "p_use", "E_use", "Z_2state", "chi_OU", "F_OU",
    "V_min_location", "V_curvature", "V_depth", "H_StM", "regime_label",
    "temperature_group", "fit_valid",
]
participant_stm = stm[export_cols].copy()
participant_stm.to_csv(OUT_DIR / "participant_ds4_stm_parameters.csv", index=False)
print(f"Wrote participant_ds4_stm_parameters.csv ({len(participant_stm)} rows)")

comparison.to_csv(OUT_DIR / "ds4_ds5_stm_comparison.csv", index=False)
print("Wrote ds4_ds5_stm_comparison.csv")


In [ ]:
summary_row = {
    "n_total": len(stm),
    "n_valid": int(stm["fit_valid"].sum()),
    "median_beta_OU": float(valid["beta_OU"].median()),
    "median_T_OU": float(valid["T_OU"].median()),
    "median_chi_OU": float(valid["chi_OU"].median()),
    "median_tau_hat": float(valid["tau_hat"].median()),
    "r_HStM_Hemp": float(r_hstm_hemp),
    "r_activity_persistence_beta": float(r_persistence_beta),
}
ds4_stm_summary = pd.DataFrame([summary_row])
ds4_stm_summary.to_csv(OUT_DIR / "ds4_stm_summary.csv", index=False)
print(ds4_stm_summary.T)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(valid["beta_OU"], bins=40, color="#4C72B0", edgecolor="white")
axes[0].set_xlabel("beta_OU")
axes[0].set_ylabel("count")
axes[0].set_title("beta_OU distribution")

axes[1].hist(valid["T_OU"], bins=40, color="#C44E52", edgecolor="white")
axes[1].set_xlabel("T_OU")
axes[1].set_title("T_OU distribution")

fig.tight_layout()
fig.savefig(FIG_DIR / "fig1_ds4_beta_distribution.png", dpi=150)
plt.close(fig)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
tau_vals = valid["tau_hat"]
bins = np.logspace(np.log10(tau_vals.min()), np.log10(tau_vals.max()), 40)
ax.hist(tau_vals, bins=bins, color="#55A868", edgecolor="white")
ax.set_xscale("log")
ax.set_xlabel("tau_hat (days, log scale)")
ax.set_ylabel("count")
ax.set_title("DS4 StM: relaxation time distribution (daily-scale)")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig2_ds4_relaxation_time.png", dpi=150)
plt.close(fig)


In [ ]:
sample = valid.sample(n=8, random_state=42)
fig, ax = plt.subplots(figsize=(8, 6))
for _, row in sample.iterrows():
    v = v_of_x(x_grid, row["theta_hat_OU"], row["mu_hat_OU"])
    ax.plot(x_grid, v, alpha=0.7, label=f"theta={row['theta_hat_OU']:.2f}")
ax.set_xlabel("x")
ax.set_ylabel("V(x)")
ax.set_title("DS4 StM: individual OU potential wells, sample participants")
ax.legend(fontsize=7, ncol=2)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig3_ds4_vx_sample.png", dpi=150)
plt.close(fig)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(valid["mu_hat_OU"], valid["beta_OU"], c=valid["tau_hat"],
                  s=valid["chi_OU"] * 3 + 10, cmap="viridis", alpha=0.7)
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label("tau_hat (days)")
ax.set_xlabel("mu_hat_OU (= p_use)")
ax.set_ylabel("beta_OU")
ax.set_title("DS4 StM: beta_OU vs mu_hat_OU, colored by tau, sized by chi_OU")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig4_ds4_beta_xmean_scatter.png", dpi=150)
plt.close(fig)


In [ ]:
summary_lines = []
summary_lines.append("# DS4 StM -- Model Run Summary")
summary_lines.append("")
summary_lines.append("**Notebook:** `51_ds4_statistical_mechanics.ipynb`  ")
summary_lines.append("**Blueprint:** `outputs/reports/stage4/step1_modeling/ds4_stm_blueprint.md`  ")
summary_lines.append("**Run date:** " + pd.Timestamp.now().strftime("%Y-%m-%d"))
summary_lines.append("")
summary_lines.append("## Summary")
summary_lines.append("")
summary_lines.append("```")
summary_lines.append(ds4_stm_summary.T.to_string())
summary_lines.append("```")
summary_lines.append("")
summary_lines.append("## DS4 vs DS5 (descriptive only, not directly comparable)")
summary_lines.append("")
summary_lines.append("```")
summary_lines.append(comparison.to_string())
summary_lines.append("```")
summary_lines.append("")
summary_lines.append("## Timescale caveat (carried forward from StoP)")
summary_lines.append("")
summary_lines.append("beta_OU/chi_OU/H_StM use daily-scale theta_hat_OU/sigma_hat_OU "
                     "only. StoP's own weekly sensitivity check found a meaningfully "
                     "different relaxation time at weekly resolution -- not a "
                     "timescale-independent property.")

with open(OUT_DIR / "model_run_summary.md", "w") as f:
    f.write("\n".join(summary_lines) + "\n")
print("Wrote model_run_summary.md")


## Interpretation and limits

**Read the cells above for live numbers.**

- FLAG A (DS5's approximate Boltzmann bridge) is resolved here: `beta_OU=2theta/sigma^2` is exact for DS4's OU process.
- `H_StM` is not an independent measurement -- it is InT's `H_SDE` under a different name, joined directly (see blueprint SS0).
- All quantities here are daily-timescale specific (FLAG 6) -- do not present `beta_OU` as a timescale-independent "behavioral temperature."
- Individual `V(x)` is exact for the OU model assumption, not a direct measurement of a true behavioral energy landscape (FLAG III).
- DS4 vs DS5 `beta` comparison is descriptive only -- different underlying SDEs, not directly comparable in magnitude.

### DS4 weakness flags

FLAG I (no condition), FLAG II (self-selected users / attrition), FLAG III (V(x) not causally identified), FLAG IV (real N=641, not all 1,039), FLAG V (behavioral proxy), FLAG VI (daily-timescale specific) -- see `ds4_stm_blueprint.md` SS8 for full text.


## Forward connections

| Connection | Mechanism | Status |
|---|---|---|
| DS4 StM -> Phenotyping | beta_OU, tau_hat, chi_OU as three phenotype dimensions | Pending -- DS4 Step 2, after NeT (54) |
| DS4 StM -> Paper | OU equations (SS4 of blueprint) not yet in the conceptual paper | Before submission |

**Next step:** DS4 NeT (54) -- independent of StoP/InT/DyS/StM (only needs sequence engineering), completes the 5-family DS4 Step 1 scope.
